# B2-023 — Practice p24

*55 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** challenge · **Difficulty:** advanced · **Time budget:** 55 minutes  
**Concepts:** stable-diffusion

## Task

Audit this latent conditional-diffusion pipeline. `LatentDenoiser`, `encode`, `decode`, `LATENT_SCALE`, `q_sample`, and `sample_loop` follow the p20 contracts; `train_on(model, z0, labels)` runs p20's training loop on the latents and labels it is given (without encoding them again); `sample(guide, w, targets)` runs the reverse loop with `guide` as the noise prediction and returns one final latent per label in `targets` (the audited listing below omits `targets`; read its calls as `targets = labels[heldout][:48]`); your repair must pass `targets` explicitly; `hit_rate(x, y)` is p20's nearest-centre hit rate.

```text
 1  rows = generative_data.load_rows("latent4d", range(240))
 2  labels = generative_data.load_labels("latent4d", range(240))
 3  cond_ids = list(generative_data.TRAIN_IDS["latent4d"]) + list(generative_data.HELDOUT_IDS["latent4d"])[:24]
 4  z0 = encode(rows[cond_ids])
 5  torch.manual_seed(20261015); model = LatentDenoiser()
 6  train_on(model, z0, labels[cond_ids])
 7  def guide(z, ts, y, w):
 8      e_u = model(z, ts, y, torch.zeros(len(y), dtype=torch.bool))
 9      e_c = model(z, ts, y, torch.ones(len(y), dtype=torch.bool))
10      return e_u - w * (e_c - e_u)
11  heldout = list(generative_data.HELDOUT_IDS["latent4d"])
12  best_w = max((0.5, 1.0, 1.5), key=lambda w: hit_rate(decode(sample(guide, w) / LATENT_SCALE), labels[heldout][:48]))
13  x = decode(sample(guide, best_w) / LATENT_SCALE)
14  report(hit_rate(x, labels[heldout][:48]))          # "final held-out score"
```

(A) **Leakage in the conditioning set.** List every train/held-out leakage edge with its line numbers, state which held-out information reaches which training or selection decision, and classify each as data leakage or selection leakage.

(B) **Guidance sign.** Identify the line with the sign-flipped guidance term. With the literals `U` and `C` in the supplied cell, implement `cfg_combine` (p12 contract) and `flipped_guidance(u, c, w) = u - w * (c - u)`; assert that they agree at $w=0$, that at $w=1$ the correct one returns `C` while the flipped one returns `[[-0.2, -0.6], [0.5, 1.5]]` (`ATOL = RTOL = 1e-6`), and explain which way the flipped version moves samples relative to the class.

(C) **Latent scaling factor.** Identify the two lines whose scaling disagrees (answer with lines 4 and 13; note that line 12 repeats line 13's decode expression). Define `encode`, `decode`, and `LATENT_SCALE` exactly as in p20, and let `R = generative_data.train_tensor("latent4d")`. Show that the per-coordinate standard deviations of `encode(R)` exceed `2.0` while those of `encode(R) * LATENT_SCALE` are below `0.7`, and that for the class centres `C3 = generative_data.class_centers()`, `decode(encode(C3) / LATENT_SCALE)` equals `4 * C3` (`ATOL = RTOL = 1e-6`). Explain what the pipeline's samples look like as a result, and why the training loss gives no warning.

(D) **Repair.** Write the repaired pipeline as a numbered listing (markdown) and as code that runs without training: the conditioning set is exactly the training IDs; latents are scaled once at encoding and unscaled once before decoding; guidance uses `cfg_combine`; $w$ is chosen on a validation subset carved from the training IDs: use the last 16 training IDs of each class as validation, giving 144 training and 48 validation rows; held-out rows are evaluated once at the end. Then wrap `encode` (recording every row it receives) while you build the repaired training latents, and assert that none of the recorded rows hashes into the held-out part of `generative_data.ROW_SHA256["latent4d"]` and that the recorded hashes are exactly the hashes of your training-minus-validation rows.

**Required answer-check assertions.** The probes in (B) and (C), the hash check in (D), and the repaired training latents' shape (your stated train-minus-validation size by 2).

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

U = torch.tensor([[0.4, -0.2], [0.0, 1.0]])
C = torch.tensor([[1.0, 0.2], [-0.5, 0.5]])
ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.